In [0]:
import zipfile
import json
import os
import pytz
from pyspark.sql.types import IntegerType, DoubleType, LongType, StructType
from pyspark.sql.utils import AnalysisException
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from datetime import datetime
from functools import reduce
from delta.tables import DeltaTable
from pyspark.sql.window import Window 

In [0]:
# ==============================
# Config Azure Storage
# ==============================

STORAGE = "stgbbb"
CONTAINER_BRONZE = "bronze"
CONTAINER_SILVER = "silver"

storage_key = dbutils.secrets.get(scope="bbb", key="secret-stg-bbb")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE}.dfs.core.windows.net",
    storage_key
)

schema_path = "abfss://config@stgbbb.dfs.core.windows.net/schema"

In [0]:

def unzip_files(file):
    try:
        zip_name = file.name
        local_zip_path = f"/dbfs/tmp/{zip_name}"
        local_extract_path = f"/dbfs/tmp/extract_{zip_name.replace('.zip', '')}"
        
        dbutils.fs.cp(file.path, "file:" + local_zip_path)
        
        with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
            zip_ref.extractall(local_extract_path)
            
    except Exception as e:
        print(e)
    
    return local_extract_path
 

In [0]:

def schema_existe(nome_tabela):
    try:
        dbutils.fs.ls(f"{schema_path}/{nome_tabela}.json")
        return True
    except AnalysisException:
        return False

In [0]:
def save_schema(df, nome_tabela):
    schema_json = df.schema.json()
    caminho = f"{schema_path}/{nome_tabela}.json"
    dbutils.fs.put(caminho, schema_json, overwrite=True)

In [0]:
def enforce_schema(df, nome_tabela):

    if not schema_existe(nome_tabela):
        save_schema(df, nome_tabela)
        return df

    """Lê o JSON do Lake e força o DF a seguir os tipos definidos."""
    caminho = f"{schema_path}/{nome_tabela}.json"
    
    # 1. Ler o arquivo JSON do Storage
    json_str = dbutils.fs.head(caminho)
    stored_schema = StructType.fromJson(json.loads(json_str))
    
    stored_cols_map = {f.name: f.dataType for f in stored_schema}
    
    select_expr = []
    has_schema_changed = False
    
    for col_name in df.columns:
        
        if col_name in stored_cols_map:
            # A coluna JÁ EXISTIA: Forçamos o tipo antigo (Enforcement) para segurança
            target_type = stored_cols_map[col_name]
            # Ex: Se era Date e veio String, tenta converter. Se falhar, vira Null (Safe Cast)
            select_expr.append(F.col(col_name).cast(target_type).alias(col_name))
        else:
            # A coluna é NOVA: Deixamos passar como veio (Evolution)
            select_expr.append(F.col(col_name))
            has_schema_changed = True
    
    df_final = df.select(select_expr)
    
    if has_schema_changed:
        save_schema(df_final, nome_tabela)
        
    return df_final

In [0]:

def clean_and_fill(df, columns, replacement="não informado"):
    from pyspark.sql.functions import when, col, trim
    """
    Transforma strings vazias em nulos e preenche nulos com um valor padrão.
    
    Args:
        df: DataFrame do Spark.
        columns: Lista de colunas para tratar (ex: ["regiao", "status"]).
        replacement: Texto que substituirá o nulo.
    """
    for column in columns:
        # Primeiro: trata espaços e transforma "" em None (nulo)
        df = df.withColumn(
            column, 
            when(trim(col(column)) == "", None).otherwise(col(column))
        )
    
    # Segundo: preenche todos os nulos das colunas selecionadas com o texto desejado
    df = df.fillna(replacement, subset=columns)   

    return df

In [0]:

def lower_and_upper(df, columns):
    for column in columns:
        df = df.withColumn(column, F.lower(F.col(column)))
    df = df.toDF(*[c.upper() for c in df.columns])
    return df

In [0]:
def limpar_sujeira(df, columns):
    from pyspark.sql import functions as F
    
    # Se 'columns' for uma única string, transforma em lista para o loop funcionar
    if isinstance(columns, str):
        columns = [columns]
        
    df_limpo = df
    
    # Itera sobre cada coluna da lista
    for col_name in columns:
        # Filtra apenas o que é número (0-9)
        df_limpo = df_limpo.filter(F.col(col_name).rlike("^[0-9]+$"))
    
    return df_limpo

In [0]:
def tratamento(df, columns):
    df = clean_and_fill(df, columns)
    df = lower_and_upper(df, columns)
    return df

In [0]:
# ==============================
# Data automática (Brasil)
# ==============================

fuso_br = pytz.timezone("America/Sao_Paulo")
agora = datetime.now(fuso_br)

# Partição automática (D-2)
particao_hoje = f"ano={agora.year}/mes={agora.month:02d}/dia={agora.day:02d}"

print("Partição usada:", particao_hoje)

In [0]:
# ==============================
# Funções reutilizáveis
# ==============================

def get_bronze_path(dataset):
    return (
        f"abfss://{CONTAINER_BRONZE}@{STORAGE}.dfs.core.windows.net/"
        f"balancacomercial/{particao_hoje}/{dataset}/"
    )

def get_bronze_path_cnpj(dataset):
    return (
        f"abfss://{CONTAINER_BRONZE}@{STORAGE}.dfs.core.windows.net/"
        f"cnpj/{particao_hoje}/{dataset}/"
    )

def get_silver_path(dataset):
    return (
        f"abfss://{CONTAINER_SILVER}@{STORAGE}.dfs.core.windows.net/"
        f"balancacomercial/{dataset}/"
    )

def read_bronze(dataset):
    path = get_bronze_path(dataset)
    print(f"Lendo Bronze: {path}")
    return spark.read.parquet(path, encoding="lattinn")

def read_bronze_cnpj(dataset):
    path = get_bronze_path_cnpj(dataset)
    print(f"Lendo Bronze: {path}")
    return spark.read.parquet(path)    

In [0]:
#funcao criada para popular os datasets de comex
def load_all_datasets(lista):
    dfs = {}
    for dataset in lista:
        dfs[dataset] = read_bronze(dataset)
    return dfs


#funcao criada para popular os datasets de cnpj
def load_all_datasets_cnpj(lista):
    dfs = {}
    for dataset in lista:
        dfs[dataset] = read_bronze_cnpj(dataset)
    return dfs

In [0]:
def process_silver_layer(dfs_dict, cast_config, business_keys, sk_name):
    """
    Função genérica para unir múltiplos DataFrames e aplicar transformações da camada Silver.
    Ajustada para funcionar com um único DataFrame ou múltiplos, e para lidar com esquemas divergentes.
    Inclui remoção de acentos para colunas do tipo 'string_unaccented' e deduplicação baseada nas business_keys.
    
    Args:
        dfs_dict (dict): Dicionário de DataFrames { "nome": df }.
        cast_config (dict): Dicionário mapeando coluna para seu tipo Spark (ex: {"CO_ANO": "int"}).
                            Para remover acentos, use {"COLUNA_TEXTO": "string_unaccented"}.
        business_keys (list): Lista de colunas que formam a chave de negócio para deduplicação e SK.
        sk_name (str): Nome da coluna de Surrogate Key a ser criada.
        
    Returns:
        DataFrame: DataFrame único, transformado e deduplicado.
    """
    
    if not dfs_dict:
        raise ValueError("O dicionário 'dfs_dict' não pode estar vazio.")
    
    dfs_list = list(dfs_dict.values())
    
    # 1. União dos DataFrames e Reconciliação de Esquema
    if len(dfs_list) == 1:
        df_unified = dfs_list[0]
    else:
        all_columns = []
        for df in dfs_list:
            all_columns.extend(df.columns)
        unique_columns = sorted(list(set(all_columns))) # Garante ordem consistente

        aligned_dfs = []
        for df in dfs_list:
            for col_name in unique_columns:
                if col_name not in df.columns:
                    df = df.withColumn(col_name, F.lit(None))
            aligned_dfs.append(df.select(unique_columns))
        
        df_unified = reduce(DataFrame.unionAll, aligned_dfs)
    
    # 2. Aplicação dinâmica de Casts e Remoção de Acentos
    for col_name, col_type in cast_config.items():
        if col_name not in df_unified.columns:
            continue # Pula se a coluna não existe no DataFrame unificado

        if col_type == "date":
            df_unified = df_unified.withColumn(
                col_name,
                F.when(
                    (F.col(col_name).isNotNull()) &
                    (F.col(col_name) != "") &
                    (F.col(col_name) != "00000000") &
                    (F.length(F.col(col_name)) == 8),
                    F.to_date(F.col(col_name), "yyyyMMdd")
                ).otherwise(None)
            )
        elif col_type in ["float", "double"]:
            df_unified = df_unified.withColumn(
                col_name,
                F.when(
                    (F.col(col_name).isNotNull()) &
                    (F.col(col_name) != ""),
                    F.regexp_replace(
                        F.regexp_replace(F.col(col_name), r"\\.", ""),
                        ",", "."
                    ).cast(col_type)
                ).otherwise(None)
            )
        elif col_type in ["int", "bigint"]:
            df_unified = df_unified.withColumn(
                col_name,
                F.when(
                    (F.col(col_name).isNotNull()) &
                    (F.col(col_name) != ""),
                    F.regexp_replace(F.col(col_name), r",.*", "")
                    .cast(col_type)
                ).otherwise(None)
            )
        elif col_type == "string_unaccented":
            # Aplica a remoção de acentos e cast para string, convertendo para minúsculas
            df_unified = df_unified.withColumn(
                col_name,
                F.lower(
                    F.regexp_replace(
                        F.regexp_replace(
                            F.regexp_replace(
                                F.regexp_replace(
                                    F.regexp_replace(
                                        F.regexp_replace(F.col(col_name), "[ÁÀÂÃÄ]", "A"),
                                    "[ÉÈÊË]", "E"),
                                "[ÍÌÎÏ]", "I"),
                            "[ÓÒÔÕÖ]", "O"),
                        "[ÚÙÛÜ]", "U"),
                    "[Ç]", "C")
                ).cast("string") 
            )
        else:
            # Caso normal para outros tipos ou string padrão sem remoção de acentos
            df_unified = df_unified.withColumn(
                col_name,
                F.col(col_name).cast(col_type)
            )
    
    # 3. Deduplicação baseada nas Business Keys
    # Garante que cada registro na Silver seja único pelas suas chaves de negócio.
    # Isso é crucial para dimensões e para evitar explosão de dados em joins futuros.
    if not business_keys:
        # Se não há business_keys, não podemos garantir unicidade, mas a função pode continuar
        print("Aviso: Nenhuma business_key fornecida para deduplicação. O DataFrame pode conter duplicatas.")
        df_deduplicated = df_unified
    else:
        df_deduplicated = df_unified.dropDuplicates(business_keys)

    # 4. Criação da Surrogate Key (SK) via MD5
    if not business_keys:
        raise ValueError("A lista 'business_keys' não pode estar vazia para criar a Surrogate Key.")

    df_transformed = df_deduplicated.withColumn(
        sk_name,
        F.md5(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in business_keys]))
    )
    
    # 5. Organização final: SK na primeira posição
    cols = [sk_name] + [c for c in df_transformed.columns if c != sk_name]
    
    return df_transformed.select(cols)

In [0]:

def save_silver_incremental(df, table_name, primary_keys):
    """
    Salva os dados na camada Silver usando o formato Delta com lógica de MERGE (Upsert).
    Isso evita duplicatas e é muito mais performático que joins manuais em Parquet.
    """
    print(f"\nIniciando Processamento Silver: {table_name}")
    
    # Caminho da tabela (Azure Data Lake Storage Gen2)
    silver_path = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/{table_name}/"
    
    # 1. Limpeza e Deduplicação Interna (Batch Atual)
    # Assumindo que a função clean_and_fill já existe no seu ambiente
    string_cols = [c for c, t in df.dtypes if t == "string"]
    df = clean_and_fill(df, string_cols)
    
    # Remove duplicatas que possam vir no mesmo lote de processamento
    df = df.dropDuplicates(primary_keys)
    
    # 2. Operação Delta Lake
    try:
        # Verifica se o caminho já contém uma tabela Delta válida
        if DeltaTable.isDeltaTable(spark, silver_path):
            print(f"Tabela Delta encontrada em {table_name}. Realizando MERGE...")
            
            delta_table = DeltaTable.forPath(spark, silver_path)
            
            # Construção dinâmica da condição de junção para o Merge
            # Ex: "target.SK_ID = updates.SK_ID AND target.CO_ANO = updates.CO_ANO"
            merge_condition = " AND ".join([f"target.{k} = updates.{k}" for k in primary_keys])
            
            delta_table.alias("target").merge(
                df.alias("updates"),
                merge_condition
            ).whenNotMatchedInsertAll().execute()
            
            print(f"Merge finalizado com sucesso para {table_name}")
        else:
            raise Exception("Caminho existe mas não é uma tabela Delta")

    except Exception as e:
        # Se a tabela não existir ou ocorrer erro na validação Delta, fazemos o overwrite inicial
        print(f"Criando nova tabela Delta (Primeira Carga ou Erro): {table_name}")
        
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(silver_path)
            
        print(f"Tabela Delta criada com sucesso em: {silver_path}")


In [0]:

def save_silver_incremental_cnpj(df, table_name, primary_keys):
    """
    Salva os dados na camada Silver usando o formato Delta com lógica de MERGE (Upsert).
    Isso evita duplicatas e é muito mais performático que joins manuais em Parquet.
    """
    print(f"\nIniciando Processamento Silver: {table_name}")
    
    # Caminho da tabela (Azure Data Lake Storage Gen2)
    silver_path = f"abfss://silver@stgbbb.dfs.core.windows.net/cnpj/{table_name}/"

    # 1. Limpeza e Deduplicação Interna (Batch Atual)
    # Assumindo que a função clean_and_fill já existe no seu ambiente
    string_cols = [c for c, t in df.dtypes if t == "string"]
    df = clean_and_fill(df, string_cols)
    
    # Remove duplicatas que possam vir no mesmo lote de processamento
    df = df.dropDuplicates(primary_keys)
    
    # 2. Operação Delta Lake
    try:
        # Verifica se o caminho já contém uma tabela Delta válida
        if DeltaTable.isDeltaTable(spark, silver_path):
            print(f"Tabela Delta encontrada em {table_name}. Realizando MERGE...")
            
            delta_table = DeltaTable.forPath(spark, silver_path)
            
            # Construção dinâmica da condição de junção para o Merge
            # Ex: "target.SK_ID = updates.SK_ID AND target.CO_ANO = updates.CO_ANO"
            merge_condition = " AND ".join([f"target.{k} = updates.{k}" for k in primary_keys])
            
            delta_table.alias("target").merge(
                df.alias("updates"),
                merge_condition
            ).whenNotMatchedInsertAll().execute()
            
            print(f"Merge finalizado com sucesso para {table_name}")
        else:
            raise Exception("Caminho existe mas não é uma tabela Delta")

    except Exception as e:
        # Se a tabela não existir ou ocorrer erro na validação Delta, fazemos o overwrite inicial
        print(f"Criando nova tabela Delta (Primeira Carga ou Erro): {table_name}")
        
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(silver_path)
            
        print(f"Tabela Delta criada com sucesso em: {silver_path}")


In [0]:
def save_hive_table(df, target_path, pk):
    if not spark.catalog.tableExists(target_path):
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_path)
    else:
        target_table = DeltaTable.forName(spark, target_path)
        target_table.alias("target").merge(
            df.alias("source"),
            f"target.{pk} = source.{pk}"
        ).whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()